# Taurus/Sigma Oviz Figure Tutorial

This notebook shows the compact workflow used to build the current Taurus/Sigma `oviz` figure.

**Packages needed:** `oviz`, `numpy`, `pandas`, `plotly`, `astropy`, `galpy`, `scipy`, `matplotlib`, `IPython`.

**Data needed:**
- Taurus/Sigma CSV: `taurus_core_sigma_age-Feb-2025-v2.csv`
- Hunt cluster velocities: `cluster_velocities_jan2026.csv`
- Chronos ages: `hunt_sample_chronos_ages_multiprocessing_feb_2026.csv`
- Cluster family catalog: `cluster_sample_data.csv`
- Edenhofer dust FITS cube: `mean_and_std_xyz.fits`
  - Zenodo record: [Edenhofer et al. 1.25 kpc/2 kpc dust-map dataset](https://zenodo.org/records/8212070)
  - Direct Cartesian map download: [mean_and_std_xyz.fits](https://zenodo.org/records/8212070/files/mean_and_std_xyz.fits?download=1) (large, about 15.7 GB)
- Internet access in the browser for Herschel/Planck sky backgrounds.

Put these files in one data folder, then set `DATA_DIR` below to that folder.


In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import FileLink, IFrame, display

from oviz import Animate3D, Trace, TraceCollection

# Find the repo root whether the notebook is launched from the repo or examples/.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
REPO_ROOT = next(path for path in candidates if (path / "oviz").exists() and (path / "tests").exists())
sys.path.insert(0, str(REPO_ROOT))

# Import the maintained Taurus script so this tutorial stays consistent with the figure runner.
script_path = REPO_ROOT / "tests" / "taurus_core_sigma_figure.py"
spec = importlib.util.spec_from_file_location("taurus_core_sigma_figure", script_path)
taurus_fig = importlib.util.module_from_spec(spec)
spec.loader.exec_module(taurus_fig)

taurus_fig.configure_runtime_environment()

## Paths

Set the folder where your files live. For a shared copy, put the four CSV files and the downloaded Edenhofer FITS file in one folder, then set `DATA_DIR` to that folder. You can also use the matching `OVIZ_*` environment variables.


In [ ]:
# User-editable roots. Environment variables let this run on another machine without editing the notebook.
HOME_DIR = Path(os.environ.get("OVIZ_HOME_DIR", Path.home())).expanduser()
DATA_DIR = Path(os.environ.get("OVIZ_DATA_DIR", HOME_DIR / "Downloads")).expanduser()
RESEARCH_DIR = Path(os.environ.get("OVIZ_RESEARCH_DIR", HOME_DIR / "Desktop" / "astro_research")).expanduser()

# Prefer files in DATA_DIR. Keep fallbacks for the original local repo layout.
default_edenhofer = DATA_DIR / "mean_and_std_xyz.fits"
if not default_edenhofer.exists():
    default_edenhofer = DATA_DIR / "mean_and_std_xyz-2.fits"

default_hunt_cluster_catalog = DATA_DIR / "cluster_velocities_jan2026.csv"
if not default_hunt_cluster_catalog.exists():
    default_hunt_cluster_catalog = (
        RESEARCH_DIR / "supernovae_map_work" / "clusters" / "vels_output" / "2026-02-04" / "cluster_velocities_jan2026.csv"
    )

# Input files. Override any single file path with its matching environment variable if needed.
input_csv = Path(os.environ.get("OVIZ_TAURUS_SIGMA_CSV", DATA_DIR / "taurus_core_sigma_age-Feb-2025-v2.csv")).expanduser()
taurus_fig.CHRONOS_AGES_PATH = Path(os.environ.get("OVIZ_CHRONOS_AGES", DATA_DIR / "hunt_sample_chronos_ages_multiprocessing_feb_2026.csv")).expanduser()
taurus_fig.CLUSTER_SAMPLE_DATA_PATH = Path(os.environ.get("OVIZ_CLUSTER_SAMPLE_DATA", DATA_DIR / "cluster_sample_data.csv")).expanduser()
taurus_fig.EDENHOFER_FITS_PATH = Path(os.environ.get("OVIZ_EDENHOFER_FITS", default_edenhofer)).expanduser()
taurus_fig.HUNT_CLUSTER_CATALOG_PATH = Path(os.environ.get("OVIZ_HUNT_CLUSTER_CATALOG", default_hunt_cluster_catalog)).expanduser()

# Output goes inside the repo by default, but you can put it in DATA_DIR or HOME_DIR if preferred.
output_html = Path(os.environ.get("OVIZ_TAURUS_OUTPUT_HTML", REPO_ROOT / "examples" / "taurus_core_sigma" / "taurus_core_sigma_figure_tutorial.html")).expanduser()

required = {
    "Taurus/Sigma CSV": input_csv,
    "Hunt cluster catalog": taurus_fig.HUNT_CLUSTER_CATALOG_PATH,
    "Chronos ages": taurus_fig.CHRONOS_AGES_PATH,
    "Cluster family catalog": taurus_fig.CLUSTER_SAMPLE_DATA_PATH,
    "Edenhofer FITS cube": taurus_fig.EDENHOFER_FITS_PATH,
}

# Fail early with clear missing-file messages.
for label, path in required.items():
    taurus_fig.require_existing_path(path, label)

output_html.parent.mkdir(parents=True, exist_ok=True)
output_html


## Load And Clip Data

Load the cluster catalogs and the Taurus/Sigma CSV. The bulk Taurus/Sigma clusters use median positions and velocities for tracebacks. The individual stars are shown only at `t=0`. Everything is clipped to the `+/-500 pc` box.


In [ ]:
# Load context clusters, Taurus/Sigma bulk clusters, and individual Taurus/Sigma stars.
df_hunt_60, df_hunt_young, family_frames = taurus_fig.load_cluster_context()
taurus_bulk = taurus_fig.load_taurus_bulk_clusters(input_csv)
taurus_stars = taurus_fig.load_taurus_star_sample(input_csv)

# Keep only the local 1 kpc box shown in this figure: x/y/z within +/-500 pc.
df_hunt_60 = taurus_fig.clip_dataframe_to_scene_bounds(df_hunt_60)
df_hunt_young = taurus_fig.clip_dataframe_to_scene_bounds(df_hunt_young)
family_frames = [
    (label, taurus_fig.clip_dataframe_to_scene_bounds(frame), color)
    for label, frame, color in family_frames
]
taurus_bulk = taurus_fig.clip_dataframe_to_scene_bounds(taurus_bulk)
taurus_stars = taurus_fig.clip_dataframe_to_scene_bounds(taurus_stars)

# Used only for view metadata; the camera is still Sun/LSR centered.
taurus_center = {
    "x": float(np.nanmedian(taurus_bulk["x"])),
    "y": float(np.nanmedian(taurus_bulk["y"])),
    "z": float(np.nanmedian(taurus_bulk["z"])),
}

len(taurus_bulk), len(taurus_stars), len(df_hunt_60), len(df_hunt_young)

## Build Traces

`Trace` objects are the moving 3D datasets. Bulk Taurus/Sigma clusters include `U,V,W`, so `oviz` can move them backward in time. Individual stars are added later as static traces, so they do not have velocities.


In [ ]:
# The Sun is a simple reference point with no traceback velocity.
sun = pd.DataFrame({
    "name": ["Sun"],
    "family": ["Sun"],
    "age_myr": [8000.0],
    "U": [0.0], "V": [0.0], "W": [0.0],
    "x": [0.0], "y": [0.0], "z": [27.0],
    "n_stars": [1],
})

# Bulk Taurus/Sigma clusters get velocities, so oviz can compute traceback positions.
taurus_trace = Trace(
    taurus_bulk[["x", "y", "z", "U", "V", "W", "name", "age_myr", "n_stars"]],
    data_name=taurus_fig.TAURUS_TRACE_NAME,
    min_size=8.0,
    max_size=8.0,
    color="#56ff96",
    opacity=0.25,
    marker_style="circle",
    show_tracks=False,
)

# Context traces are the same cluster samples/families used in the current figure.
traces = TraceCollection([
    Trace(sun, data_name="Sun", min_size=5, max_size=5, color="yellow"),
    taurus_trace,
    Trace(df_hunt_60, data_name="Clusters (< 60 Myr)", min_size=0, max_size=7, color="grey", opacity=0.34),
    Trace(df_hunt_young, data_name="Young Clusters (< 15 Myr)", min_size=1.25, max_size=7, color="red", opacity=0.86),
    *[
        Trace(frame, data_name=label, min_size=0, max_size=7, color=color, opacity=0.95)
        for label, frame, color in family_frames
        if not frame.empty
    ],
])

## Groups And Axes

Set the legend groups and axis style. This figure uses normal `x/y/z` axes, not galactic mode. Ticks are spaced every 200 pc.


In [ ]:
# Legend groupings control which traces are visible from the group dropdown.
family_names = [label for label, frame, _color in family_frames if not frame.empty]
trace_groupings = {
    "Taurus Sigma": ["Sun", taurus_fig.TAURUS_TRACE_NAME],
    "Taurus + Context": [
        "Sun", taurus_fig.TAURUS_TRACE_NAME,
        "Clusters (< 60 Myr)", "Young Clusters (< 15 Myr)", *family_names,
    ],
    "Cluster Families": ["Sun", taurus_fig.TAURUS_TRACE_NAME, *family_names],
    "Young Clusters": ["Sun", taurus_fig.TAURUS_TRACE_NAME, "Young Clusters (< 15 Myr)"],
    "<60 Myr Cluster Sample": ["Sun", taurus_fig.TAURUS_TRACE_NAME, "Clusters (< 60 Myr)"],
}

plot_3d = Animate3D(
    data_collection=traces,
    xyz_ranges=(
        taurus_fig.SCENE_BOUNDS["x"],
        taurus_fig.SCENE_BOUNDS["y"],
        taurus_fig.SCENE_BOUNDS["z"],
    ),
    figure_theme="dark",
    trace_grouping_dict=trace_groupings,
)

# Clean grey box axes at 200 pc spacing.
for axis_name, axis_title in (("xaxis", "x (pc)"), ("yaxis", "y (pc)"), ("zaxis", "z (pc)")):
    axis_layout = plot_3d.figure_layout_dict["scene"][axis_name]
    axis_layout.update(
        title=axis_title,
        visible=True,
        showline=True,
        showgrid=False,
        zeroline=False,
        linecolor="#777777",
        linewidth=1.0,
        tickfont={"size": 14, "color": "#777777", "family": "Helvetica"},
        title_font={"size": 18, "color": "#777777", "family": "Helvetica"},
        dtick=200,
        nticks=6,
    )

## Dust, Sky, And Static Stars

Add the Edenhofer dust cube and crop it to the `+/-500 pc` box before downsampling. The initial state sets Herschel SPIRE as the default sky background, with Planck behind it.


In [ ]:
# Edenhofer dust volume. clip_bounds truly crops the cube before downsampling.
edenhofer_volume = {
    "name": "Edenhofer+2024 Dust",
    "path": str(taurus_fig.EDENHOFER_FITS_PATH),
    "hdu": "MEAN",
    "clip_bounds": {axis: list(bounds) for axis, bounds in taurus_fig.SCENE_BOUNDS.items()},
    "max_resolution": 512,
    "max_resolution_cap": 512,
    "opacity": 1.0,
    "samples": 200,
    "alpha_coef": 200,
    "vmin": 0,
    "vmax": 0.0385,
    "colormap": "Greys",
    "supports_show_all_times": True,
    "co_rotate_with_frame": True,
}

# Static t=0 traces: individual stars and labels. Stars have no velocity, so they do not trace back.
static_traces = taurus_fig.build_taurus_static_traces(taurus_bulk, taurus_stars)

# Initial state controls camera, point scale, sky layers, and default volume visibility.
initial_state = taurus_fig.main_style_initial_state(
    compact_payload=True,
    taurus_center=taurus_center,
)

## Render

Create the standalone Three.js HTML file. The final lines enforce the box limits, match the point-size scaling from `main_figure`, and hide labels by default.


In [ ]:
# Build the interactive Three.js figure.
# time goes backward in Myr; oviz integrates cluster bulk velocities for each frame.
figure = plot_3d.make_plot(
    time=np.round(np.arange(0, -66, -1), 1),
    show=False,
    save_name=None,
    static_traces=static_traces,
    static_traces_times=[[0.0], [0.0]],
    static_traces_legendonly=False,
    focus_group=None,
    fade_in_time=8,
    fade_in_and_out=False,
    show_age_kde_inset=True,
    age_kde_bandwidth_myr=2,
    include_spiral_arms=False,
    galactic_mode=False,
    show_galactic_guides=False,
    camera_zoom_factor=5,
    show_gc_line=False,
    show_milky_way_model=False,
    renderer="threejs",
    enable_sky_panel=True,
    volumes=[edenhofer_volume],
    threejs_initial_state=initial_state,
)

# Match the current Taurus figure post-processing.
taurus_fig.clip_scene_spec_to_scene_bounds(figure.scene_spec)
figure.scene_spec["point_size_reference_span_pc"] = taurus_fig.MAIN_FIGURE_POINT_SIZE_REFERENCE_SPAN_PC
taurus_fig.default_trace_off(figure.scene_spec, taurus_fig.TAURUS_LABEL_TRACE_NAME)

figure.write_html(str(output_html))
display(FileLink(output_html))

## Preview

Open the HTML from the link. The inline preview is optional and may be slow because the figure is large.


In [ ]:
# Optional inline preview. If the notebook UI struggles with a large HTML file, open the FileLink above instead.
IFrame(src=str(output_html), width="100%", height=720)